# Phase 2: Target & Feature Definition

**Course:** Advanced AI Techniques (AAI)  
**Project:** AI-Enabled Micro-channel Battery Thermal Management System for Electric Vehicles (KJS-CES-02)  
**Dataset:** FreeTwinEV P45B 1s4p Battery Module Measurement  

This notebook establishes the formal mathematical definitions and reproducible pipeline for:
1. Target Variables: Absolute Future Peak Temperature $T_{\text{cell,max}}(t+h)$ vs. Future Temperature Rise $\Delta T_h(t)$
2. Prediction Horizons: Primary $h=60\text{s}$ (12 steps) and Secondary $h=300\text{s}$ (60 steps)
3. Core Baseline Features: Compact, physically measurable inputs available at decision time $t$
4. Derived Features: Non-linear Joule heating proxies, rolling load integrals, and thermal velocity
5. Ablation Features: Downstream calorimetric signals ($\text{temp\_outlet}$) isolated for separate sensitivity testing
6. Run-Boundary Audit: Empirical proof that shifting and rolling operations never cross experimental boundaries

**Strict Rule:** No ML models are trained in this phase.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root))

from src.freetwinev_eda import load_all_runs
from src.freetwinev_targets import process_all_runs_targets
from src.freetwinev_features import (
    compute_features_all_runs,
    get_feature_schema,
    CORE_FEATURES,
    DERIVED_FEATURES,
    ABLATION_FEATURES
)

all_data_dir = project_root / 'data' / 'candidate_freetwinev' / '1s4p Battery Module Measurement' / 'AllData'
print('Loading 14 physical runs from:', all_data_dir)
runs = load_all_runs(str(all_data_dir))
print(f'Loaded {len(runs)} experimental runs successfully.')

## 1. Feature Taxonomy and Specification Schema

The complete feature schema defining categories, transformations, window lengths, and leakage status.

In [ ]:
df_schema = get_feature_schema()
print(f'Total features defined: {len(df_schema)}')
print(f'- Core Baseline Features: {len(CORE_FEATURES)}')
print(f'- Derived Features: {len(DERIVED_FEATURES)}')
print(f'- Ablation Features: {len(ABLATION_FEATURES)}')
df_schema[['Feature', 'Category', 'Unit', 'Availability', 'Window', 'Leakage_Status']]

## 2. Target Construction with Strict Run-Boundary Enforcement

We construct future targets for $h = 60\text{s}$ (12 steps) and $h = 300\text{s}$ (60 steps).  
Each experiment is processed independently so that shifting never crosses from one experiment to the next.

In [ ]:
horizons = {'60s': 12, '300s': 60}
runs_with_targets = process_all_runs_targets(runs, horizons=horizons)
runs_featured = compute_features_all_runs(runs_with_targets)

# Load run boundary audit results
df_boundary = pd.read_csv(project_root / 'results' / 'freetwinev' / 'run_boundary_audit.csv')
df_boundary

## 3. Statistical Analysis of Candidate Targets

Comparison of Absolute Future Temperature $T_{\text{cell,max}}(t+h)$ vs Future Temperature Rise $\Delta T_h(t)$.

In [ ]:
df_tgt_stats = pd.read_csv(project_root / 'results' / 'freetwinev' / 'target_summary_stats.csv')
df_tgt_stats

## 4. Feature Summary Statistics across All Observations

Summary statistics for all Core, Derived, and Ablation features.

In [ ]:
df_feat_stats = pd.read_csv(project_root / 'results' / 'freetwinev' / 'feature_summary_stats.csv')
df_feat_stats

## 5. Feature-Target Correlations

Correlations between input features and candidate targets, highlighting the physical predictive signal for $\Delta T_h$.

In [ ]:
df_corr = pd.read_csv(project_root / 'results' / 'freetwinev' / 'feature_target_correlations.csv')
df_corr.sort_values(by='Corr_DeltaT_60s', key=abs, ascending=False)